# 03. Batch Inference and Cloud Spanner Audit Queue Write
**Cymbal Financial Fraud Detection Pipeline**

This notebook loads the trained `RandomForestClassifier` pipeline model, runs batch scoring on all incoming unlabeled transactions (`is_fraud IS NULL`), isolates high-risk transactions ($	ext{fraud probability} \ge 0.50$), and writes them directly into the **Cloud Spanner** operational compliance review queue (`SparkEvalFraudReviewQueue`).

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, udf
from pyspark.sql.types import DoubleType
from pyspark.ml import PipelineModel

spark = SparkSession.builder \
    .appName("Cymbal-Fraud-Batch-Inference") \
    .getOrCreate()

project_id = os.getenv("PROJECT_ID", "cymbal-fraud-detection")
dataset_name = os.getenv("DATASET_NAME", "transactions_dataset_evals")
models_bucket = os.getenv("MODELS_BUCKET", f"{project_id}-models")
spanner_instance = os.getenv("SPANNER_INSTANCE", "cymbal-fraud")
spanner_database = os.getenv("SPANNER_DATABASE", "fraud-db")
spanner_table = "SparkEvalFraudReviewQueue"
model_path = f"gs://{models_bucket}/fraud_model"

print(f"Target Cloud Spanner: {spanner_instance} / {spanner_database} / {spanner_table}")

In [ ]:
# Load the trained ML Pipeline Model
try:
    model = PipelineModel.load(model_path)
    print(f"Loaded model from Cloud Storage: {model_path}")
except Exception as e:
    print(f"Loading local fallback model: {e}")
    model = PipelineModel.load("models/fraud_model")
    print("Loaded local model successfully.")

In [ ]:
# Load unlabeled records from BigQuery enriched table
try:
    enriched_df = spark.read.format("bigquery") \
        .option("table", f"{project_id}.{dataset_name}.enriched_transactions") \
        .load()
except Exception as e:
    import duckdb
    con = duckdb.connect("data/cymbal_fraud.duckdb")
    pdf = con.execute("SELECT * FROM enriched_transactions").df()
    enriched_df = spark.createDataFrame(pdf)

# Filter only incoming unlabeled transactions
unlabeled_df = enriched_df.filter(col("is_fraud").isNull())
print(f"Total unlabeled incoming transactions to score: {unlabeled_df.count():,}")

In [ ]:
# Run Batch Scoring
predictions = model.transform(unlabeled_df)

# Extract probability of the positive class (is_fraud = 1.0)
extract_fraud_prob = udf(lambda v: float(v[1]) if v is not None and len(v) > 1 else 0.0, DoubleType())
scored_df = predictions.withColumn("fraud_probability", extract_fraud_prob(col("probability")))

# Filter for High-Risk Alerts (fraud_probability >= 0.50)
high_risk_df = scored_df.filter(col("fraud_probability") >= 0.50)
print(f"High-Risk Anomalous Transactions Identified: {high_risk_df.count():,}")

In [ ]:
# Format and align schema for Cloud Spanner SparkEvalFraudReviewQueue
spanner_columns = [
    "transaction_id",
    "amount",
    "currency",
    "device_id",
    "ip_address",
    "merchant_mcc",
    "payee_id",
    "payment_method",
    "payor_id",
    "status",
    "timestamp",
    "payor_name",
    "payor_country",
    "payor_risk_score",
    "payee_name",
    "payee_country",
    "payee_risk_score",
    "payee_category",
    "is_fraud",
    "prediction"
]

final_spanner_df = high_risk_df \
    .withColumn("prediction", col("prediction").cast("double")) \
    .withColumn("merchant_mcc", col("merchant_mcc").cast("long")) \
    .withColumn("is_fraud", col("prediction").cast("long")) \
    .select(*spanner_columns)

final_spanner_df.show(5, truncate=False)

In [ ]:
# Write High-Risk alerts directly into Cloud Spanner
try:
    final_spanner_df.write \
        .format("cloud-spanner") \
        .option("instanceId", spanner_instance) \
        .option("databaseId", spanner_database) \
        .option("table", spanner_table) \
        .mode("append") \
        .save()
    print("Successfully appended high-risk alerts to Cloud Spanner SparkEvalFraudReviewQueue.")
except Exception as e:
    print(f"Cloud Spanner write bypassed (Local Mode): {e}")
    import duckdb
    con = duckdb.connect("data/cymbal_fraud.duckdb")
    pdf_spanner = final_spanner_df.toPandas()
    con.execute("CREATE OR REPLACE TABLE SparkEvalFraudReviewQueue AS SELECT * FROM pdf_spanner")
    print(f"Successfully saved {len(pdf_spanner)} high-risk alerts to local Spanner review queue table.")